In [2]:
#importing 
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset as HFDataset
import evaluate
import pandas as pd
import numpy as np
import torch

c:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ── Load processed data ───────────────────────────────────────
syn_train = pd.read_csv("../data/processed/syn_train.csv")
syn_val   = pd.read_csv("../data/processed/syn_val.csv")
syn_test  = pd.read_csv("../data/processed/syn_test.csv")

In [4]:
#Model selection
MODEL_NAME = "xlm-roberta-base"
# MODEL_NAME = "bert-base-multilingual-cased"


In [5]:
# ── Detect GPU & set memory-efficient dtype ────────────────────
device    = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16  = torch.cuda.is_available()   # auto-enable on GPU
print(f"Using device : {device}")
print(f"FP16 enabled : {use_fp16}")
print(f"Model        : {MODEL_NAME}\n")

Using device : cuda
FP16 enabled : True
Model        : xlm-roberta-base



In [6]:
# ── Tokenizer ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["review_text"],
        truncation=True,
        padding="max_length",
        max_length=128          # 128 saves memory vs 512; increase if needed
    )

In [8]:
# ── Dataset helper ─────────────────────────────────────────────
def to_hf_dataset(df):
    return HFDataset.from_dict({
        "review_text": df["review_text"].tolist(),
        "label":       df["label"].tolist(),
        "word_count":  df["word_count"].tolist(),
        "rating":      df["rating"].tolist(),
        "slang_count": df["slang_count"].tolist(),
    })

train_hf = to_hf_dataset(syn_train).map(tokenize, batched=True)
val_hf   = to_hf_dataset(syn_val).map(tokenize,   batched=True)
test_hf  = to_hf_dataset(syn_test).map(tokenize,  batched=True)

Map: 100%|██████████| 448/448 [00:00<00:00, 10936.21 examples/s]
